In [ ]:
%sql
-- Create the catalog (top-level namespace)
CREATE CATALOG IF NOT EXISTS analytics;

In [ ]:
%sql
-- Switch to the target catalog
USE CATALOG analytics;

-- Create a new schema 
CREATE SCHEMA IF NOT EXISTS sales;

In [ ]:
%sql
-- Create a new table in the analytics.ventas schema to store cleaned NYC taxi trip data
CREATE TABLE analytics.ventas.nyctaxi_trips_clean

-- Specify Delta Lake as the storage format for reliability, performance, and ACID compliance
USING DELTA

-- Populate the table using the results of the following SELECT query (CTAS pattern)
AS
SELECT
    -- Original pickup timestamp of the trip
    tpep_pickup_datetime,

    -- Original dropoff timestamp of the trip
    tpep_dropoff_datetime,
    
    -- Calculate trip duration in minutes using pickup and dropoff timestamps
    TIMESTAMPDIFF(
        MINUTE,
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    ) AS trip_duration_minutes,
    
    -- Distance traveled during the trip (in miles)
    trip_distance,
    
    -- Compute fare per mile as a derived metric
    -- ROUND is used to limit the result to 2 decimal places for readability
    ROUND(fare_amount / trip_distance, 2) AS fare_per_mile,
    
    -- Pickup location ZIP code
    pickup_zip,

    -- Dropoff location ZIP code
    dropoff_zip
    
-- Source dataset containing raw NYC taxi trip records
FROM samples.nyctaxi.trips

-- Filter out invalid or problematic records:
-- Exclude trips with zero or negative distance to avoid division errors
WHERE trip_distance > 0

-- Exclude trips with zero or negative fare to ensure meaningful metrics
AND fare_amount > 0;

## Constraints in Delta Lake ("Data Integrity Rule")

### Enforced Constraints 

In [ ]:
%sql
CREATE TABLE analytics.sales.users (
    id INT,
    age INT
);

In [ ]:
%sql
ALTER TABLE analytics.sales.users
ADD CONSTRAINT age_positive CHECK (age >= 0);

In [ ]:
%sql
INSERT INTO analytics.sales.users
VALUES (1, 28)

In [ ]:
%sql
INSERT INTO analytics.sales.users
VALUES (1, -28)
# [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint age_positive (age >= 0) violated by row with values:  - age : -28. SQLSTATE: 23001

In [ ]:
%sql
SHOW TBLPROPERTIES analytics.sales.users